In [11]:
import pandas as pd
from pathlib import Path
import re

## PSPLIB project sample
Sample used for the model comparison.

(1) 100 projects are drawn at random from each original PSPLIB set {J120, J90, J60, J30}: 400 projects in total.

In [135]:
# Sample of 100 projects per set (400 in total)
# folder with the .sm files
df = pd.read_csv("./PSPLIB_sample/psplib_sample_combined.csv")
print('Number of files:', len(df))
results_dir = Path("./PSPLIB_sample/")
for file in results_dir.glob("*.sm"):
    if file.name not in df["FileName"].values:
        print(f"File {file.name} not found in the CSV")


Number of files: 400


(2) In each set of 100 projects, 10% are used for the hyperparameter selection of every model family and the remaining 90% for the model comparison.

In [137]:
df.groupby(['SubSet','model_selection']).count()['FileName']

SubSet  model_selection
J120    False              90
        True               10
J30     False              90
        True               10
J60     False              90
        True               10
J90     False              90
        True               10
Name: FileName, dtype: int64

## Model selection

```
For each LHS configuration:
    For each model-selection project:
        train on 80% of the replications
        evaluate on the fixed 20%
    aggregate the errors
choose the configuration with the best mean performance/rank
```

In [138]:
# Parameterisation
# Number of LHS configurations: 50
for file in Path("model_selection/hyperparameters").glob("*.csv"):
    df_temp = pd.read_csv(file)
    print(f"{file.name}: {len(df_temp)}")

lhs_ebm_parameters.csv: 50
lhs_en_parameters.csv: 50
lhs_fastai_parameters.csv: 50
lhs_rf_parameters.csv: 50
lhs_sv_parameters.csv: 50
lhs_xgb_parameters.csv: 50


In [139]:
# Projects selected in each set
# Number of projects for the hyperparameter selection:
#   40 = 10 x {J120, J90, J60, J30}
# (the IDs are in model_selection/project_selection.csv)
df_mselect = pd.read_csv("./model_selection/project_selection.csv")
df_mselect.groupby("SubSet").count()['sample']

SubSet
J120    10
J30     10
J60     10
J90     10
Name: sample, dtype: int64

In [144]:
# Number of Monte Carlo replications: 10,000
df=pd.DataFrame({'files': list(Path("montecarlo").glob("*.csv"))})
for _ in range(5):
    f= df.sample(n=1)['files'].iloc[0]
    print(f.name,pd.read_csv(f).shape)


j3045_9_simulation_01.csv (10000, 35)
j12012_3_simulation_01.csv (10000, 125)
j9025_9_simulation_01.csv (10000, 95)
j3016_5_simulation_01.csv (10000, 35)
j607_8_simulation_01.csv (10000, 65)


In [145]:
# Number of test runs (20% of 10,000)
# (the IDs are in montecarlo/test_runs.csv)
print (pd.read_csv("montecarlo/test_runs.csv").shape)

(2000, 1)


## Model evaluation

In [143]:
# Number of projects for the model evaluation: 360 (90 x {J120, J90, J60, J30})
df_meval = pd.read_csv("./model_evaluation/project_evaluation.csv")
df_meval.groupby("SubSet").count()['sample']

SubSet
J120    90
J30     90
J60     90
J90     90
Name: sample, dtype: int64